In [ ]:
# !pip install datasets
# Run this if you are runnning on Colab 

# Below resultsare tested on "NVIDIA GeForce RTX 5090" on RUN POD

# Device: CUDA
# GPU: NVIDIA GeForce RTX 5090

# ======================================================================
# PERFORMANCE SUMMARY
# ======================================================================
#   1. BASELINE FP32 TRAINING                          51.6135s (1.00x)
#   2. MIXED PRECISION (PyTorch AMP + Autocast)        35.1863s (0.68x)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 798.4 kB/s  0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 616.3/616.3 kB 1.0 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 7.6 MB/s  0:00:0036m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 3.6 MB/s  0:00:000m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 30.8 MB/s  0:00:01m0:00:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 16.3 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23/23 [datasets]/23 [datasets]ce-hub]


In [1]:
"""
Mixed Precision Training Implementation
========================================

Demonstrates mixed precision training techniques:
1. PyTorch Automatic Mixed Precision (AMP) with torch.autocast
2. Manual mixed precision with explicit casting
3. Loss scaling to prevent gradient underflow
4. Gradient accumulation with mixed precision
5. Performance comparison: FP32 vs FP32+FP16 mixed precision

Mixed precision uses lower precision (FP16/BF16) for most operations while
keeping high precision (FP32) for loss computation and weight updates.

Key benefits:
- Reduces memory usage by ~50% (FP16 = 2 bytes vs FP32 = 4 bytes)
- Speeds up computation (especially on GPUs with Tensor Cores)
- Maintains convergence quality with proper loss scaling
"""

import time
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import autocast, GradScaler
from typing import Tuple, Dict
from datasets import load_dataset, load_from_disk
from torch.utils.data import DataLoader, Dataset

# --- Model config ---
batch_size = 32
embed_dim = 768
attn_heads = 12
ffn_dim = 3072
vocab_size = 50257
seq_len = 512
num_layers = 12
NUM_TRAIN_STEPS = 100
LR = 3e-4
MAX_GRAD_NORM = 1.0
USE_REAL_DATA = True  # Set to False to use random data
DATA_CACHE_DIR = "./data/wikitext-103"


# Detect device: CUDA > MPS (macOS) > CPU
def get_device():
    """Get the best available device"""
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")


# --- Data loading utilities ---
def download_wikitext():
    """Download Wikitext-103 dataset"""
    print("  Downloading Wikitext-103...")
    try:
        dataset = load_dataset("wikitext", "wikitext-103-v1")
        dataset.save_to_disk(DATA_CACHE_DIR)
        print(f"  Saved to {DATA_CACHE_DIR}")
        return dataset
    except Exception as e:
        print(f"  Error downloading: {e}")
        return None


def load_wikitext():
    """Load Wikitext-103 from cache or download"""
    if os.path.exists(DATA_CACHE_DIR):
        print(f"  Loading cached Wikitext-103 from {DATA_CACHE_DIR}")
        return load_from_disk(DATA_CACHE_DIR)
    else:
        return download_wikitext()


class TokenizedDataset(Dataset):
    """Tokenize text data into token sequences"""
    def __init__(self, texts, vocab_size=50257, seq_len=512):
        self.texts = [t for t in texts if t.strip()]  # Filter empty
        self.vocab_size = vocab_size
        self.seq_len = seq_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        # Simple hash-based "tokenization" for demo
        # In practice, use BPE tokenizer (tiktoken, transformers)
        tokens = [hash(word) % self.vocab_size for word in text.split()]

        # Pad or truncate to seq_len
        if len(tokens) < self.seq_len:
            tokens = tokens + [0] * (self.seq_len - len(tokens))
        else:
            tokens = tokens[:self.seq_len]

        input_ids = torch.tensor(tokens[:-1], dtype=torch.long)
        targets = torch.tensor(tokens[1:], dtype=torch.long)

        return input_ids, targets


def get_data_batch(use_real=True):
    """Generator yielding batches of data"""
    if not use_real:
        # Random data
        while True:
            input_ids = torch.randint(0, vocab_size, (batch_size, seq_len))
            targets = torch.randint(0, vocab_size, (batch_size, seq_len))
            yield input_ids, targets
    else:
        # Real data from Wikitext-103
        dataset = load_wikitext()
        if dataset is None:
            print("  Falling back to random data")
            yield from get_data_batch(use_real=False)
            return

        texts = dataset['train']['text']
        wikitext_dataset = TokenizedDataset(texts, vocab_size, seq_len)
        loader = DataLoader(wikitext_dataset, batch_size=batch_size, shuffle=True)

        while True:
            for batch in loader:
                yield batch


DEVICE = get_device()


class Transformer(nn.Module):
    """Single transformer layer"""

    def __init__(self):
        super().__init__()
        self.attn_norm = nn.LayerNorm(embed_dim)
        self.attention = nn.MultiheadAttention(
            embed_dim, attn_heads, bias=False, batch_first=True
        )
        self.ffn_norm = nn.LayerNorm(embed_dim)
        self.linear1 = nn.Linear(embed_dim, ffn_dim)
        self.linear2 = nn.Linear(ffn_dim, embed_dim)

    def forward(self, x):
        residual = x
        x = self.attn_norm(x)
        x, _ = self.attention(x, x, x)
        x = x + residual
        residual = x
        x = self.ffn_norm(x)
        x = self.linear1(x)
        x = F.gelu(x)
        x = self.linear2(x)
        return x + residual


class Model(nn.Module):
    """Transformer-based model"""

    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.transformers = nn.ModuleList([Transformer() for _ in range(num_layers)])
        self.output = nn.Linear(embed_dim, vocab_size, bias=False)

    def forward(self, token_ids):
        x = self.embedding(token_ids)
        for t in self.transformers:
            x = t(x)
        return self.output(x)


class MixedPrecisionTrainer:
    """
    Trainer with mixed precision support using PyTorch AMP.
    Handles automatic casting to FP16 for forward pass with loss scaling.
    """

    def __init__(self, model: nn.Module, lr: float = 1e-2):
        self.model = model
        self.optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        # GradScaler helps prevent gradient underflow in FP16
        self.scaler = GradScaler()
        self.device = DEVICE
        self.model.to(self.device)

    def train_step(self, input_ids: torch.Tensor, targets: torch.Tensor) -> Tuple[float, float]:
        """
        Single training step with mixed precision.

        Flow:
        1. Forward pass in FP16 (autocast) → reduced memory
        2. Loss computed in FP32 (outside autocast) → numerical stability
        3. Loss scaled to prevent gradient underflow
        4. Backward pass with scaled loss
        5. Unscale gradients before optimizer step
        6. Clip gradients and update weights in FP32
        """
        input_ids = input_ids.to(self.device)
        targets = targets.to(self.device)

        self.optimizer.zero_grad()

        # Forward pass: automatically cast to FP16
        with autocast(device_type=self.device.type, dtype=torch.float16):
            logits = self.model(input_ids)
            loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))

        # Scale loss and backward
        self.scaler.scale(loss).backward()

        # Unscale before gradient clipping
        self.scaler.unscale_(self.optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(self.model.parameters(), MAX_GRAD_NORM)

        # Optimizer step with scaling
        self.scaler.step(self.optimizer)
        self.scaler.update()

        return loss.item(), grad_norm.item()

    def eval_forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        """Inference with mixed precision"""
        input_ids = input_ids.to(self.device)
        with torch.no_grad(), autocast(
            device_type=self.device.type, dtype=torch.float16
        ):
            return self.model(input_ids)


class BaselineTrainer:
    """Baseline FP32 trainer for comparison"""

    def __init__(self, model: nn.Module, lr: float = 1e-2):
        self.model = model
        self.optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        self.device = DEVICE
        self.model.to(self.device)

    def train_step(self, input_ids: torch.Tensor, targets: torch.Tensor) -> Tuple[float, float]:
        """Standard FP32 training"""
        input_ids = input_ids.to(self.device)
        targets = targets.to(self.device)

        self.optimizer.zero_grad()
        logits = self.model(input_ids)
        loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(self.model.parameters(), MAX_GRAD_NORM)
        self.optimizer.step()

        return loss.item(), grad_norm.item()

    def eval_forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        """Inference in FP32"""
        input_ids = input_ids.to(self.device)
        with torch.no_grad():
            return self.model(input_ids)


def train_with_trainer(trainer_class, model, name: str) -> Dict:
    """Train model using specified trainer"""
    torch.manual_seed(42)
    model = model.__class__()  # Fresh model

    torch.manual_seed(123)  # Same training seed
    trainer = trainer_class(model, lr=LR)

    print(f"\n{'=' * 70}")
    print(f"{name}")
    print(f"{'=' * 70}")
    print(
        f"\n  Config: {NUM_TRAIN_STEPS} steps, lr={LR}, batch={batch_size}, seq_len={seq_len}"
    )
    print(f"  Data: {'Wikitext-103' if USE_REAL_DATA else 'Random'}")
    print(f"  Trainable params: {sum(p.numel() for p in model.parameters())}")
    print(f"\n  {'Step':<6} {'Loss':<12} {'Grad Norm':<12}")
    print(f"  {'-' * 30}")

    start_time = time.time()
    data_gen = get_data_batch(use_real=USE_REAL_DATA)

    for step in range(NUM_TRAIN_STEPS):
        input_ids, targets = next(data_gen)
        loss, grad_norm = trainer.train_step(input_ids, targets)
        if step % 10 == 0:
            print(f"  {step:<6} {loss:<12.4f} {grad_norm:<12.4f}")

    total_time = time.time() - start_time

    # Inference
    print(f"\n  {'Inference:':<30}")
    with torch.no_grad():
        torch.manual_seed(999)
        test_ids = torch.randint(0, vocab_size, (1, seq_len))
        output = trainer.eval_forward(test_ids)

    return {
        "name": name,
        "total_time": total_time,
    }


def print_profiler_stats(results: Dict):
    """Print profiler statistics"""
    print(f"\n{'=' * 70}")
    print("PROFILER RESULTS")
    print(f"{'=' * 70}")
    for result in results:
        print(f"\n{result['name']}:")
        print(f"  Total time: {result['total_time']:.4f}s")
        if result["profiler"]:
            print(f"\n  Top operations:")
            print(result["profiler"].key_averages().table(sort_by="cpu_memory_usage", row_limit=10))




In [2]:
print(f"\n{'#' * 70}")
print("# MIXED PRECISION TRAINING COMPARISON")
print(f"{'#' * 70}")

base_model = Model()

# Determine device info
if torch.cuda.is_available():
    print("\n  Device: CUDA")
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    print("\n  Device: MPS (macOS)")
    print("  Metal Performance Shaders available")
else:
    print("\n  Device: CPU")
    print("  No GPU acceleration available")

results = []

# Run baseline FP32
results.append(
    train_with_trainer(BaselineTrainer, base_model, "1. BASELINE FP32 TRAINING")
)

# Run mixed precision with autocast
results.append(
    train_with_trainer(
        MixedPrecisionTrainer,
        base_model,
        "2. MIXED PRECISION (PyTorch AMP + Autocast)",
    )
)

# Performance summary
print(f"\n{'=' * 70}")
print("PERFORMANCE SUMMARY")
print(f"{'=' * 70}")
baseline_time = results[0]["total_time"]
for result in results:
    time_ratio = result["total_time"] / baseline_time
    print(f"  {result['name']:<50} {result['total_time']:.4f}s ({time_ratio:.2f}x)")

print(f"\n{'#' * 70}")
print("# TRAINING COMPLETE")
print(f"{'#' * 70}\n")


######################################################################
# MIXED PRECISION TRAINING COMPARISON
######################################################################

  Device: CUDA
  GPU: NVIDIA GeForce RTX 5090

1. BASELINE FP32 TRAINING

  Config: 100 steps, lr=0.0003, batch=32, seq_len=512
  Data: Wikitext-103
  Trainable params: 162212352

  Step   Loss         Grad Norm   
  ------------------------------
  Loading cached Wikitext-103 from ./data/wikitext-103
  0      12.5881      80.0293     
  10     1.8678       9.0214      
  20     1.5565       2.4853      
  30     1.0194       0.4457      
  40     0.9779       0.3180      
  50     1.3347       0.3733      
  60     1.4349       0.3616      
  70     1.0985       0.3500      
  80     1.3729       0.3077      
  90     0.8404       0.2339      

  Inference:                    

2. MIXED PRECISION (PyTorch AMP + Autocast)

  Config: 100 steps, lr=0.0003, batch=32, seq_len=512
  Data: Wikitext-103
  Trainabl